In [1]:
import sys
sys.path.append("../src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from stats_toolkit import pearson_r, correlation_summary

rr = pd.read_csv("../data/raw/gaia_dr3_rrlyrae_sample_ri20M_raw.csv")
rr["period"] = rr["pf"].fillna(rr["p1_o"])
rr["period_type"] = np.where(rr["pf"].notna(), "fundamental", "overtone")
rr["log_period"] = np.log10(rr["period"])

print(rr.shape, "| duplicate source_ids:", rr["source_id"].duplicated().sum())
print(rr["period_type"].value_counts())
print()
print(rr.isna().mean().round(3))
print()
print(rr[["log_period", "peak_to_peak_g", "phot_g_mean_mag", "bp_rp"]].describe().round(3))

(2000, 12) | duplicate source_ids: 0
period_type
fundamental    1312
overtone        688
Name: count, dtype: int64

source_id              0.000
pf                     0.344
p1_o                   0.646
peak_to_peak_g         0.000
num_clean_epochs_g     0.000
random_index           0.000
phot_g_mean_mag        0.000
bp_rp                  0.063
parallax_over_error    0.026
period                 0.000
period_type            0.000
log_period             0.000
dtype: float64

       log_period  peak_to_peak_g  phot_g_mean_mag     bp_rp
count    2000.000        2000.000         2000.000  1874.000
mean       -0.337           0.568           18.227     1.168
std         0.138           0.269            1.828     0.730
min        -0.686           0.102           11.244    -1.247
25%        -0.470           0.358           16.782     0.632
50%        -0.290           0.495           18.474     0.862
75%        -0.227           0.763           19.747     1.604
max        -0.028           1.39

In [2]:
rr2 = rr.dropna(subset=["log_period", "peak_to_peak_g"])
groups = {"pooled": rr2,
          "fundamental": rr2[rr2["period_type"] == "fundamental"],
          "overtone": rr2[rr2["period_type"] == "overtone"]}

rows = []
for name, d in groups.items():
    s = correlation_summary(d["log_period"], d["peak_to_peak_g"])
    sp = stats.pearsonr(d["log_period"], d["peak_to_peak_g"])
    rows.append({"group": name, "n": s["n"], "r": s["r"], "r2": s["r2"],
                 "ci_low": s["ci95"][0], "ci_high": s["ci95"][1],
                 "scipy r": sp[0],
                 "spearman": stats.spearmanr(d["log_period"], d["peak_to_peak_g"])[0]})
print(pd.DataFrame(rows).round(3).to_string(index=False))

f = groups["fundamental"]["peak_to_peak_g"]
o = groups["overtone"]["peak_to_peak_g"]
print()
print("mean amplitude: fundamental", round(f.mean(), 3), "| overtone", round(o.mean(), 3))
print("Welch t:", stats.ttest_ind(f, o, equal_var=False))

      group    n      r    r2  ci_low  ci_high  scipy r  spearman
     pooled 2000  0.411 0.169   0.374    0.447    0.411     0.334
fundamental 1312 -0.411 0.169  -0.455   -0.365   -0.411    -0.462
   overtone  688  0.222 0.049   0.150    0.292    0.222     0.220

mean amplitude: fundamental 0.681 | overtone 0.354
Welch t: TtestResult(statistic=np.float64(40.283867487354), pvalue=np.float64(2.5204921076144806e-255), df=np.float64(1851.0748789945972))
